In [1]:
# -*- coding: utf-8 -*-
"""Bước 1: tính định lý Bayes bằng tay, không cần thư viện học máy."""

def bayes(tien_nghiem, kha_nang, kha_nang_nguoc):
    """Trả về xác suất hậu nghiệm P(A|B)."""
    tu_so = kha_nang * tien_nghiem
    mau_so = tu_so + kha_nang_nguoc * (1 - tien_nghiem)
    return tu_so / mau_so

print("Vi du 1: xet nghiem benh")
print(" Ty le mac benh 1 phan tram, xet nghiem dung 99 phan tram.")
p = bayes(tien_nghiem=0.01, kha_nang=0.99, kha_nang_nguoc=0.01)
print(f" Duong tinh thi kha nang co benh that = {p:.4f}")
print()

print("Cung xet nghiem do, doi ty le mac benh:")
print(" Ty le mac benh | Hau nghiem khi duong tinh")
for ty_le in [0.001, 0.01, 0.05, 0.10, 0.50]:
    p_val = bayes(ty_le, 0.99, 0.01)
    print(f" {ty_le:12.3f} | {p_val:18.4f}")
print()

print("Vi du 2: hai day chuyen san xuat")
print(" Day A lam 80 phan tram san pham, ty le loi 1 phan tram.")
print(" Day B lam 20 phan tram san pham, ty le loi 5 phan tram.")
p_b = bayes(tien_nghiem=0.20, kha_nang=0.05, kha_nang_nguoc=0.01)
print(f" Nhat duoc san pham loi, kha nang no tu day B = {p_b:.4f}")
print()

print("Kiem lai bang cach dem tren 1000 san pham:")
loi_a = 800 * 0.01
loi_b = 200 * 0.05
print(f" Day A: 800 san pham, {loi_a:.0f} cai loi")
print(f" Day B: 200 san pham, {loi_b:.0f} cai loi")
tong_loi = loi_a + loi_b
print(f" Ty le tu day B = {loi_b} / {tong_loi} = {loi_b / tong_loi:.4f}")

Vi du 1: xet nghiem benh
 Ty le mac benh 1 phan tram, xet nghiem dung 99 phan tram.
 Duong tinh thi kha nang co benh that = 0.5000

Cung xet nghiem do, doi ty le mac benh:
 Ty le mac benh | Hau nghiem khi duong tinh
        0.001 |             0.0902
        0.010 |             0.5000
        0.050 |             0.8390
        0.100 |             0.9167
        0.500 |             0.9900

Vi du 2: hai day chuyen san xuat
 Day A lam 80 phan tram san pham, ty le loi 1 phan tram.
 Day B lam 20 phan tram san pham, ty le loi 5 phan tram.
 Nhat duoc san pham loi, kha nang no tu day B = 0.5556

Kiem lai bang cach dem tren 1000 san pham:
 Day A: 800 san pham, 8 cai loi
 Day B: 200 san pham, 10 cai loi
 Ty le tu day B = 10.0 / 18.0 = 0.5556


In [3]:
# -*- coding: utf-8 -*-
"""Bước 2: đọc kho tin nhắn và biến văn bản thành bảng số."""
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

df = pd.read_csv("data/tin_nhan.csv")
print("Kich thuoc bang:", df.shape)
print()
print("So tin theo nhan (1 la rac, 0 la thuong):")
print(df["la_rac"].value_counts())
print()
print("Ty le tin rac:", round(df["la_rac"].mean(), 4))
print()

print("Ba tin nhan dau tien:")
for i in range(3):
    print(f" [{df.la_rac[i]}] {df.noi_dung[i]}")
print()

# token_pattern mặc định bỏ từ 1 ký tự, dùng \S+ để tách theo khoảng trắng
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])

print("Kich thuoc bang dem tu:", X.shape)
print("So tu trong tu dien:", len(vec.vocabulary_))
print()

dem = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
dem["la_rac"] = df["la_rac"]
tong = dem.groupby("la_rac").sum().T
tong.columns = ["thuong", "rac"]

print("Muoi tu xuat hien nhieu nhat trong tin rac:")
print(tong.sort_values("rac", ascending=False).head(10))
print()
print("Muoi tu xuat hien nhieu nhat trong tin thuong:")
print(tong.sort_values("thuong", ascending=False).head(10))

Kich thuoc bang: (300, 2)

So tin theo nhan (1 la rac, 0 la thuong):
la_rac
0    180
1    120
Name: count, dtype: int64

Ty le tin rac: 0.4

Ba tin nhan dau tien:
 [0] cáo gửi cáo thầy nhóm cáo gửi có cho
 [1] miễn giảm ngay hạn miễn tay nhé thưởng cho nhanh
 [0] cho bạn mai gửi nhóm bài chiều

Kich thuoc bang dem tu: (300, 40)
So tu trong tu dien: 40

Muoi tu xuat hien nhieu nhat trong tin rac:
       thuong  rac
tay         3   48
ngay        4   47
nhanh       9   44
chót        6   43
giảm        6   42
lúc        39   42
quà         8   41
click       5   40
phí         6   40
miễn        4   39

Muoi tu xuat hien nhieu nhat trong tin thuong:
       thuong  rac
thầy       74    7
bài        73    4
cáo        72    8
chiều      68    8
ạ          65    7
tài        64   11
tập        63    6
gửi        61   10
lớp        60    2
nhóm       58    7


In [4]:
# -*- coding: utf-8 -*-
"""Bước 3: huấn luyện bộ lọc thư rác và chấm điểm nó."""
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print("So tin de hoc :", X_train.shape[0])
print("So tin de kiem tra:", X_test.shape[0])
print()

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("Ma tran nham lan")
print(f" TN = {tn:2d} | thu thuong duoc giu dung")
print(f" FP = {fp:2d} | thu thuong bi xoa nham")
print(f" FN = {fn:2d} | thu rac lot luoi")
print(f" TP = {tp:2d} | thu rac bi chan dung")
print()

print(f"Accuracy  = {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision = {precision_score(y_test, y_pred):.4f}")
print(f"Recall    = {recall_score(y_test, y_pred):.4f}")
print()

tin_moi = [
    "trung thuong khuyen mai nhanh tay click ngay",
    "hop nhom bao cao chieu mai phong 305",
    "ban gui giup em tai lieu bai tap nhe",
]
X_moi = vec.transform(tin_moi)
p_rac = mo_hinh.predict_proba(X_moi)[:, 1]

print("Thu tren ba tin nhan tu go:")
for tin, p in zip(tin_moi, p_rac):
    nhan = "RAC" if p >= 0.5 else "thuong"
    print(f" [{nhan:6s}] p = {p:.4f} <- {tin}")

So tin de hoc : 225
So tin de kiem tra: 75

Ma tran nham lan
 TN = 45 | thu thuong duoc giu dung
 FP =  0 | thu thuong bi xoa nham
 FN =  4 | thu rac lot luoi
 TP = 26 | thu rac bi chan dung

Accuracy  = 0.9467
Precision = 1.0000
Recall    = 0.8667

Thu tren ba tin nhan tu go:
 [RAC   ] p = 0.9992 <- trung thuong khuyen mai nhanh tay click ngay
 [thuong] p = 0.0983 <- hop nhom bao cao chieu mai phong 305
 [thuong] p = 0.0719 <- ban gui giup em tai lieu bai tap nhe


In [5]:
# -*- coding: utf-8 -*-
"""Bước 4: tự cài Naive Bayes trên kho nhỏ để thấy vấn đề số không."""
from collections import Counter

kho = [
    ("trung thuong ngay", 1),
    ("khuyen mai giam gia", 1),
    ("trung thuong click", 1),
    ("hop nhom chieu mai", 0),
    ("bai tap nhom nop mai", 0),
    ("thay gui tai lieu", 0),
]

tu_dien = sorted({t for cau, _ in kho for t in cau.split()})
V = len(tu_dien)
print(f"Tu dien co {V} tu:")

dem = {0: Counter(), 1: Counter()}
tong = {0: 0, 1: 0}
for cau, nhan in kho:
    for t in cau.split():
        dem[nhan][t] += 1
        tong[nhan] += 1

def xac_suat(tu, nhan, alpha):
    return (dem[nhan][tu] + alpha) / (tong[nhan] + alpha * V)

def diem(cau, nhan, alpha):
    p = 0.5  # Tiên nghiệm cân bằng
    for t in cau.split():
        if t in tu_dien:
            p *= xac_suat(t, nhan, alpha)
    return p

cau_thu = "trung thuong bai tap"
print(f"Tin nhan thu: \"{cau_thu}\"")

for alpha in [0.0, 1.0]:
    d_rac = diem(cau_thu, 1, alpha)
    d_thuong = diem(cau_thu, 0, alpha)
    ket = "rac" if d_rac > d_thuong else "thuong"
    print(f"alpha = {alpha}")
    print(f"  diem lop rac    = {d_rac:.10f}")
    print(f"  diem lop thuong = {d_thuong:.10f}")
    print(f"  ket luan        = {ket}")
    print()

Tu dien co 18 tu:
Tin nhan thu: "trung thuong bai tap"
alpha = 0.0
  diem lop rac    = 0.0000000000
  diem lop thuong = 0.0000000000
  ket luan        = thuong

alpha = 1.0
  diem lop rac    = 0.0000073212
  diem lop thuong = 0.0000021656
  ket luan        = rac



In [7]:
# -*- coding: utf-8 -*-
"""Bước 5: Naive Bayes cho số liệu liên tục, đặt cạnh hồi quy logistic."""
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

for ten, mo_hinh in [("Naive Bayes     ", GaussianNB()),
                      ("Hoi quy logistic", LogisticRegression())]:
    mo_hinh.fit(X_train, y_train)
    acc = accuracy_score(y_test, mo_hinh.predict(X_test))
    print(f"{ten}: accuracy = {acc:.4f}")
print()

gnb = GaussianNB().fit(X_train, y_train)
print("Nhung gi Naive Bayes hoc duoc")
print(f" {'':16s} {'gio_on':>10s} {'diem_giua_ky':>14s}")
for i, ten_lop in enumerate(["Lop 0 (rot)   ", "Lop 1 (qua)   "]):
    tb = gnb.theta_[i]
    dl = gnb.var_[i] ** 0.5
    print(f" {ten_lop} trung binh | {tb[0]:10.2f} {tb[1]:14.2f}")
    print(f"              do lech   | {dl[0]:10.2f} {dl[1]:14.2f}")
print()

ty_le = ", ".join(f"{float(v):.4f}" for v in gnb.class_prior_)
print(f"Ty le tung lop trong tap hoc: {ty_le}")

Naive Bayes     : accuracy = 0.9000
Hoi quy logistic: accuracy = 0.9000

Nhung gi Naive Bayes hoc duoc
                      gio_on   diem_giua_ky
 Lop 0 (rot)    trung binh |       8.18           3.99
              do lech   |       5.29           1.92
 Lop 1 (qua)    trung binh |      20.03           6.70
              do lech   |       5.81           1.50

Ty le tung lop trong tap hoc: 0.4111, 0.5889


In [11]:
# bai1
# -*- coding: utf-8 -*-
p_n = [0.5, 0.3, 0.2]      # Tiên nghiệm các nhóm
p_q_given_n = [0.95, 0.70, 0.30]  # Khả năng qua môn từng nhóm

# Tính tử số Bayes cho từng nhóm: P(Q|Ni) * P(Ni)
tu_so = [p_q_given_n[i] * p_n[i] for i in range(3)]
mau_so = sum(tu_so)

posteriors = [tu / mau_so for tu in tu_so]
for i, p in enumerate(posteriors, 1):
    print(f"Xac suat thuoc nhom {i} khi da qua mon: {p:.4f}")

print("Tong xac suat: {sum(posteriors):.4f}")

Xac suat thuoc nhom 1 khi da qua mon: 0.6376
Xac suat thuoc nhom 2 khi da qua mon: 0.2819
Xac suat thuoc nhom 3 khi da qua mon: 0.0805
Tong xac suat: {sum(posteriors):.4f}


In [12]:
# bai2
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

mo_hinh = MultinomialNB()
mo_hinh.fit(X, y)

# feature_log_prob_[1] là log-xác suất của lớp rác, [0] là lớp thường
diff = mo_hinh.feature_log_prob_[1] - mo_hinh.feature_log_prob_[0]
tu_dien = vec.get_feature_names_out()

df_tu = pd.DataFrame({"tu": tu_dien, "chenh_lech": diff})
print("5 tu to cao tin rac manh nhat:")
print(df_tu.sort_values("chenh_lech", ascending=False).head(5))
print("\n5 tu to cao tin thuong manh nhat:")
print(df_tu.sort_values("chenh_lech", ascending=True).head(5))

5 tu to cao tin rac manh nhat:
       tu  chenh_lech
21    mãi    3.203435
30    tay    2.905600
22   ngay    2.661837
20   miễn    2.479516
33  trúng    2.374155

5 tu to cao tin thuong manh nhat:
       tu  chenh_lech
18    lớp   -2.612187
9      em   -2.561756
0     bài   -2.294553
1     báo   -2.274074
28  phòng   -2.274074


In [13]:
#bai3
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

for alpha in [0.001, 0.1, 1.0, 10.0, 100.0]:
    model = MultinomialNB(alpha=alpha)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print(f"Alpha = {alpha:6.3f} | Accuracy = {accuracy_score(y_test, y_pred):.4f} | Precision = {precision_score(y_test, y_pred):.4f} | Recall = {recall_score(y_test, y_pred):.4f}")

Alpha =  0.001 | Accuracy = 0.9467 | Precision = 1.0000 | Recall = 0.8667
Alpha =  0.100 | Accuracy = 0.9467 | Precision = 1.0000 | Recall = 0.8667
Alpha =  1.000 | Accuracy = 0.9467 | Precision = 1.0000 | Recall = 0.8667
Alpha = 10.000 | Accuracy = 0.9467 | Precision = 1.0000 | Recall = 0.8667
Alpha = 100.000 | Accuracy = 0.9467 | Precision = 1.0000 | Recall = 0.8667


In [14]:
# bai4
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

df = pd.read_csv("data/tin_nhan.csv")
X_text = df["noi_dung"].values
y = df["la_rac"].values

X_train_txt, X_test_txt, y_train, y_test = train_test_split(
    X_text, y, test_size=0.25, random_state=17, stratify=y
)

# 1. Xây dựng từ điển từ tập train
tu_dien = sorted({t for cau in X_train_txt for t in cau.split()})
V = len(tu_dien)
tu2idx = {t: i for i, t in enumerate(tu_dien)}

# 2. Đếm số lần xuất hiện
dem = {0: np.zeros(V), 1: np.zeros(V)}
tong = {0: 0, 1: 0}
p_prior = {0: np.mean(y_train == 0), 1: np.mean(y_train == 1)}

for cau, nhan in zip(X_train_txt, y_train):
    for t in cau.split():
        if t in tu2idx:
            dem[nhan][tu2idx[t]] += 1
            tong[nhan] += 1

alpha = 1.0
# Tính log-likelihood
log_prob = {}
for nhan in [0, 1]:
    log_prob[nhan] = np.log((dem[nhan] + alpha) / (tong[nhan] + alpha * V))

# 3. Dự đoán
def predict_one(cau):
    scores = {}
    for nhan in [0, 1]:
        s = np.log(p_prior[nhan])
        for t in cau.split():
            if t in tu2idx:
                s += log_prob[nhan][tu2idx[t]]
        scores[nhan] = s
    return 1 if scores[1] > scores[0] else 0

y_pred = [predict_one(cau) for cau in X_test_txt]
print(f"Accuracy tu viet: {accuracy_score(y_test, y_pred):.4f}")

Accuracy tu viet: 0.9467


In [15]:
# bai5
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

model = MultinomialNB()
model.fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))
print(f"MultinomialNB tren so lieu lien tuc accuracy = {acc:.4f}")

MultinomialNB tren so lieu lien tuc accuracy = 0.8000


In [16]:
# bai6
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("data/tin_nhan.csv")
# Tạo bộ dữ liệu mất cân bằng nặng: 180 tin thường, 20 tin rác đầu tiên
df_thuong = df[df["la_rac"] == 0]
df_rac = df[df["la_rac"] == 1].head(20)
df_moi = pd.concat([df_thuong, df_rac]).reset_index(drop=True)

vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df_moi["noi_dung"])
y = df_moi["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

model = MultinomialNB()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall rac: {recall_score(y_test, y_pred, pos_label=1):.4f}")

Accuracy : 0.9800
Recall rac: 0.8000
